# Решения: Практика: RFM плюс производные признаки

**Для преподавателя.** Секционный эталон урока и ДЗ; до сдачи ученикам не показывать.


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (Path(name), Path("../../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в ../../data")

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## Урок. 1. Базовый RFM

In [ ]:
merged=orders.merge(payments,on="order_id",validate="one_to_one")
ref_date = merged["order_purchase_timestamp"].max()
rfm = (merged.groupby("customer_id").agg(
    last_purchase=("order_purchase_timestamp", "max"),
    Frequency=("order_id", "nunique"), Monetary=("payment_value", "sum")
).reset_index())
rfm["Recency"] = (ref_date - rfm["last_purchase"]).dt.days
rfm = rfm[["customer_id", "Recency", "Frequency", "Monetary"]]
assert len(rfm)==778


## Урок. 2. Бинарная оплата

In [ ]:
merged["is_card"]=(merged["payment_type"]=="credit_card").astype(int)
assert set(merged["is_card"])=={0,1}


## Урок. 3. Доля card

In [ ]:
card_share=merged.groupby("customer_id")["is_card"].mean().rename("share_card")
rfm_plus=rfm.merge(card_share,on="customer_id",validate="one_to_one")
assert rfm_plus["share_card"].between(0,1).all()


## Урок. 4. Средний срок доставки

In [ ]:
merged["days_to_deliver"]=(merged["order_delivered_customer_date"]-merged["order_purchase_timestamp"]).dt.days
delivery_mean=merged.groupby("customer_id")["days_to_deliver"].mean()
rfm_plus=rfm_plus.merge(delivery_mean.rename("avg_days_to_deliver"),on="customer_id",how="left")
assert rfm_plus["avg_days_to_deliver"].notna().sum()>700


## Урок. 5. Штат клиента

In [ ]:
rfm_plus=rfm_plus.merge(customers[["customer_id","customer_state"]],on="customer_id",validate="one_to_one")
assert len(rfm_plus)==778


## Урок. 6. Корреляция F и M

In [ ]:
corr_fm=float(rfm_plus["Frequency"].corr(rfm_plus["Monetary"]))
CORR_NOTE="Положительная корреляция означает, что в этих данных клиенты с большим числом заказов обычно имеют большую суммарную оплату. Это не причинная связь: Monetary по определению накапливается с заказами, а размер чека и период наблюдения создают дополнительную вариацию."
assert -1<=corr_fm<=1


## Урок. 7. Масштабированный score

In [ ]:
scored=rfm_plus.copy()
scored["score"]=scored["Frequency"]/scored["Frequency"].mean()+scored["Monetary"]/scored["Monetary"].mean()-scored["Recency"]/scored["Recency"].mean()
assert np.isfinite(scored["score"]).all()


## Урок. 8. Контракт RFM+

In [ ]:
checks={"unique":rfm_plus["customer_id"].is_unique,"frequency":rfm_plus["Frequency"].ge(1).all(),"money":rfm_plus["Monetary"].gt(0).all(),"share":rfm_plus["share_card"].between(0,1).all(),"no_churn":"churn" not in rfm_plus}
assert set(checks.values())=={True}


## ДЗ. 1. Median Monetary по штату

In [ ]:
state_median=rfm_plus.groupby("customer_state")["Monetary"].median()
assert state_median.gt(0).all()


## ДЗ. 2. Квантили признаков

In [ ]:
quantiles=rfm_plus[["Recency","Frequency","Monetary"]].quantile([.25,.5,.75])
assert quantiles.shape==(3,3)


## ДЗ. 3. Top-10 score

In [ ]:
top10=scored.nlargest(10,"score")
assert top10["score"].is_monotonic_decreasing


## ДЗ. 4. Challenge: функция add_extras

In [ ]:
def add_extras(rfm_df,merged_df):
    x=merged_df.copy(); x["is_card"]=(x["payment_type"]=="credit_card").astype(int)
    x["days_to_deliver"]=(x["order_delivered_customer_date"]-x["order_purchase_timestamp"]).dt.days
    agg=x.groupby("customer_id").agg(share_card=("is_card","mean"),avg_days_to_deliver=("days_to_deliver","mean"))
    return rfm_df.merge(agg,on="customer_id",how="left",validate="one_to_one")
extra=add_extras(rfm,merged)


## ДЗ. 5. Challenge: паспорт score

In [ ]:
SCORE_NOTE="Score складывает Frequency и Monetary после деления на их средние и вычитает нормированный Recency: недавняя активность повышает результат. Масштабирование не даёт денежным единицам автоматически доминировать. Назначение — ранжирование для исследования, не прогноз. Ограничение: веса выбраны экспертно и требуют проверки бизнес-эффекта."
assert len(SCORE_NOTE)>=280
